# Learning Model Ananlysis (Deep CNN Models and Vision Transformers)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

# -----------------------------
# 1. Dataset & Transforms
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

dataset = datasets.ImageFolder(root="Preprocessed_Data", transform=transform)

train_size = int(0.7 * len(dataset))
val_size   = int(0.2 * len(dataset))
test_size  = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes = len(dataset.classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 2. Models to Compare
# -----------------------------
model_names = {
    "ResNet50": "resnet50",
    "EfficientNet-B0": "efficientnet_b0",
    "ViT-B16": "vit_base_patch16_224",
    "Swin-T": "swin_tiny_patch4_window7_224"
}

# -----------------------------
# 3. Training Function
# -----------------------------
def train_model(model, epochs=15, lr=5e-5):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        correct, total, running_loss = 0,0,0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs,1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)
        train_acc = 100*correct/total

        # Validation
        model.eval()
        correct, total = 0,0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs,1)
                correct += (preds==labels).sum().item()
                total += labels.size(0)
        val_acc = 100*correct/total

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    return model

# -----------------------------
# 4. Evaluation Function
# -----------------------------
def evaluate_model(model, model_name):
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    print(f"\n=== {model_name} Test Report ===")
    print(classification_report(y_true, y_pred, target_names=dataset.classes, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=dataset.classes, yticklabels=dataset.classes)
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    acc = np.mean(np.array(y_true)==np.array(y_pred))*100
    return acc

# -----------------------------
# 5. Run Benchmark
# -----------------------------
results = {}

for name, arch in model_names.items():
    print(f"\n----- Training {name} -----\n")
    model = timm.create_model(arch, pretrained=True, num_classes=num_classes)
    model.to(device)

    start = time.time()
    model = train_model(model, epochs=30, lr=5e-5)
    test_acc = evaluate_model(model, name)
    elapsed = (time.time()-start)/60
    params = sum(p.numel() for p in model.parameters())/1e6

    results[name] = {"Test Accuracy": test_acc, "Params (M)": params, "Time (min)": elapsed}

# -----------------------------
# 6. Show Results Table
# -----------------------------
import pandas as pd
df = pd.DataFrame(results).T
print("\n=== Benchmark Results ===")
print(df)
